In [55]:
# Import Libraries
import pandas as pd
import numpy as np
import us
from sklearn.preprocessing import StandardScaler

# Cleaning the NORS Outbreak Dataset

In [56]:
# Load the dataset
outbreak_df = pd.read_csv("/Users/ashleighcooperlindstrom/Documents/UTK/511/final_project/Raw data/NORS_outbreaks.csv", low_memory = False)

# View the data
outbreak_df.head()

,Year,Month,State,Primary Mode,Etiology,Serotype or Genotype,Etiology Status,Setting,Illnesses,Hospitalizations,Info On Hospitalizations,Deaths,Info On Deaths,Food Vehicle,Food Contaminated Ingredient,IFSAC Category,Water Exposure,Water Type,Animal Type
0,1971,2,California,Water,Copper,NaN,Confirmed,Restaurant,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
1,1971,6,Arkansas,Water,Hepatitis A,NaN,Confirmed,Store,98,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Other,NaN
2,1971,6,Missouri,Water,Unknown,NaN,Suspected,Subdivision/Neighborhood,2,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN
3,1971,6,Alabama,Water,Selenium,NaN,Confirmed,Unknown,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Individual/Private,NaN
4,1971,6,Vermont,Water,Unknown,NaN,Suspected,Community/municipality,3,NaN,NaN,0.0,NaN,NaN,NaN,NaN,Drinking water,Community,NaN


In [57]:
# Make a copy of the original dataset to clean
outbreak_clean = outbreak_df.copy()

# Rename all columns to lowercase and replace spaces with underscores
outbreak_clean.columns = outbreak_clean.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('[^0-9a-zA-Z_]', '', regex=True)

## Understand/Inspecting the Dataset

- Identify all columns (e.g., outbreak ID, state, illness type, number of cases, dates, settings).
- Check for categorical vs numeric variables.

In [58]:
# Show general info (types, missing values)
print("\nDataset Info:")
print(outbreak_clean.info())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66713 entries, 0 to 66712
Data columns (total 19 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   year                          66713 non-null  int64  
 1   month                         66713 non-null  int64  
 2   state                         66713 non-null  object 
 3   primary_mode                  66713 non-null  object 
 4   etiology                      50375 non-null  object 
 5   serotype_or_genotype          16470 non-null  object 
 6   etiology_status               50375 non-null  object 
 7   setting                       60804 non-null  object 
 8   illnesses                     66713 non-null  int64  
 9   hospitalizations              58155 non-null  float64
 10  info_on_hospitalizations      58480 non-null  object 
 11  deaths                        58785 non-null  float64
 12  info_on_deaths                58463 non-null 

In [59]:
# Show summary statistics for numeric columns
print("\nSummary Statistics:")
display(outbreak_clean.describe())


Summary Statistics:


,year,month,illnesses,hospitalizations,deaths
count,66713.000000,66713.000000,66713.000000,58155.000000,58785.000000
mean,2012.696926,5.692009,33.251660,0.825501,0.042375
std,7.546412,3.736764,1564.413045,4.164555,0.436484
min,1971.000000,1.000000,2.000000,0.000000,0.000000
25%,2009.000000,2.000000,5.000000,0.000000,0.000000
50%,2014.000000,5.000000,14.000000,0.000000,0.000000
75%,2018.000000,9.000000,30.000000,1.000000,0.000000
max,2023.000000,12.000000,403000.000000,308.000000,50.000000


### A. Categorical Variables (n=12)

**1.  States**

The state where the exposure occurred. For a single state of exposure, the state will be listed. For multiple states of exposure, "Multistate" will be listed.

Using the research we did about unknown states we will clean up and remove certain values:

In [60]:
# Look at unique values
categorical_columns = ['state']
for col in categorical_columns:
    if col in outbreak_clean.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_clean[col].unique())


Unique values in state:
['California' 'Arkansas' 'Missouri' 'Alabama' 'Vermont' 'Oregon'
 'New Jersey' 'Mississippi' 'Kentucky' 'Oklahoma' 'New Mexico'
 'North Carolina' 'New York' 'Alaska' 'Texas' 'Indiana' 'Colorado' 'Ohio'
 'Minnesota' 'Illinois' 'Florida' 'Pennsylvania' 'Washington' 'Maryland'
 'Massachusetts' 'Tennessee' 'Utah' 'Hawaii' 'Iowa' 'West Virginia'
 'Arizona' 'Virginia' 'Connecticut' 'Idaho' 'New Hampshire' 'Wisconsin'
 'Montana' 'Puerto Rico' 'Louisiana' 'Kansas' 'South Carolina' 'Maine'
 'Wyoming' 'North Dakota' 'Michigan' 'Georgia' 'South Dakota'
 'Rhode Island' 'Nevada' 'Virgin Islands' 'Multistate' 'Delaware'
 'Northern Mariana Islands' 'Nebraska' 'Guam' 'District of Columbia'
 'Republic of the Marshall Islands' 'Republic of Palau']


- Due to a low amount of data we will remove:
    - Republic of Palau
    - Virgin Islands
    - Northern Mariana Islands
    - Republic of Marshall Islands
- Map District of Columbia to Virginia since it is a state nearby

In [61]:
# List of states/territories to remove due to low data
remove_states = [
    "Republic of Palau",
    "Virgin Islands",
    "Northern Mariana Islands",
    "Republic of the Marshall Islands"
]

# Filter out the unwanted states
outbreak_clean = outbreak_clean[~outbreak_clean['state'].isin(remove_states)]

# Map District of Columbia to Virginia
outbreak_clean['state'] = outbreak_clean['state'].replace({'District of Columbia': 'Virginia'})

# Check the updated states
print(outbreak_clean['state'].value_counts())

state
Wisconsin         3912
Ohio              3757
Illinois          3481
New York          3306
Minnesota         3229
Michigan          3137
Pennsylvania      3120
Florida           3077
Virginia          3017
California        2987
Oregon            2664
Massachusetts     2450
Colorado          2020
Texas             1589
Washington        1453
North Carolina    1334
South Carolina    1251
Arizona           1225
Maryland          1076
Multistate        1069
Maine             1048
Tennessee         1045
Connecticut       1016
Iowa               948
Rhode Island       947
Alabama            904
Kentucky           865
New Hampshire      858
West Virginia      782
Kansas             781
Indiana            698
Georgia            697
Nevada             669
Utah               626
Missouri           612
Hawaii             612
Nebraska           527
Montana            446
New Mexico         411
North Dakota       341
Wyoming            321
Vermont            316
Idaho              304
Louis

**2. Primary Mode**

Primary mode of transmission.

In [62]:
categorical_columns = ['primary_mode']
for col in categorical_columns:
    if col in outbreak_clean.columns:
        print(f"\nUnique values in {col}:")
        print(outbreak_clean[col].unique())


Unique values in primary_mode:
['Water' 'Food' 'Person-to-person' 'Indeterminate/unknown'
 'Animal contact' 'Environmental contamination other than food/water']


**3. Etiology**

Genus and species of identified etiology. Multiple reported etiologies are separated by a semicolon.

In [63]:
outbreak_clean['etiology'].nunique()

787

In [64]:
# Top 20 etiologies by count
top_etiologies = outbreak_clean['etiology'].value_counts().head(20)
print(top_etiologies)

etiology
Norovirus                                        9668
Norovirus Genogroup II                           9178
Norovirus unknown                                8769
Salmonella enterica                              4123
Norovirus Genogroup I                            4108
Escherichia coli, Shiga toxin-producing          1219
Shigella sonnei                                  1172
Clostridium perfringens                           926
Legionella pneumophila                            762
Staphylococcus aureus                             626
Campylobacter jejuni                              604
Scombroid toxin                                   497
Norovirus Genogroup II;Norovirus Genogroup II     487
Unknown                                           426
Cryptosporidium unknown                           414
Bacillus cereus                                   401
Ciguatoxin                                        381
Campylobacter unknown                             298
Bacillus cereus;Clo

**4. Serotype or Genotype**

Serotype or genotype of identified etiology. Multiple reported etiologies are separated by a semicolon.

In [65]:
outbreak_clean['serotype_or_genotype'].nunique()

764

Might be a difficult variable to handle since many are a list of descriptors; will be hard to categorize. Omit this variable.

**5. Etiology Status**

Indicates whether or not the identified etiology is laboratory 'Confirmed' or 'Suspected'. Multiple reported etiologies are separated by a semicolon.

In [66]:
outbreak_clean['etiology_status'].nunique()

57

In [67]:
# Top 20 etiology_status by count
top_etiology_status = outbreak_clean['etiology_status'].value_counts().head(10)
print(top_etiology_status)

etiology_status
Confirmed                        24884
Suspected                        22969
Suspected;Suspected                791
Confirmed;Confirmed                739
Confirmed;Suspected                316
Suspected;Confirmed                302
Confirmed;Confirmed;Confirmed       77
Suspected;Suspected;Suspected       61
Confirmed;Confirmed;Suspected       32
Suspected;Suspected;Confirmed       30
Name: count, dtype: int64


Very messy variable that is hard to interpret. Omit this variable.

**6. Setting**

Setting(s) where exposure occurred. For foodborne outbreaks, the setting used is the location where the food was prepared.

In [68]:
outbreak_clean['setting'].nunique()

597

Many different settings, some have similarities, might be worth exploring ways to group them using cluster analysis.

**7. Food Vehicle**

For foodborne outbreaks only, the implicated food. Multiple implicated foods are separated by a semicolon.

In [69]:
# Top 10 food vehicles by frequency
top_food_vehicles = (
    outbreak_clean['food_vehicle']
    .value_counts()
    .head(10)
    .reset_index()
    .rename(columns={'index': 'food_vehicle', 'food_vehicle': 'count'})
)

top_food_vehicles

,count,count
0,"oysters, raw",264
1,multiple foods,208
2,"ground beef, hamburger",134
3,"salad, unspecified",130
4,chicken,115
5,"chicken, unspecified",104
6,"sandwich, submarine",92
7,"pork, BBQ",87
8,"chicken, other",85
9,"fish, mahi mahi",85


Investigate missing food_vehicle values by primary_mode

In [70]:
# Mark missing vs present food_vehicle values
outbreak_clean['food_vehicle_missing'] = outbreak_clean['food_vehicle'].apply(
    lambda x: "Missing" if pd.isna(x) or str(x).strip() == "" else "Present"
)

# Count occurrences and compute percentages by primary_mode
food_missing_summary = (
    outbreak_clean
    .groupby(['primary_mode', 'food_vehicle_missing'])
    .size()
    .reset_index(name='count')
)

food_missing_summary['percent'] = (
    food_missing_summary.groupby('primary_mode')['count']
    .transform(lambda x: 100 * x / x.sum())
)

food_missing_summary

,primary_mode,food_vehicle_missing,count,percent
0,Animal contact,Missing,655,100.000000
1,Environmental contamination other than food/water,Missing,132,100.000000
2,Food,Missing,12282,49.670401
3,Food,Present,12445,50.329599
4,Indeterminate/unknown,Missing,5677,100.000000
5,Person-to-person,Missing,32414,100.000000
6,Water,Missing,3101,100.000000


This table shows that about half of the reported food vehicles are missing within the category food within primary mode.

**8. Food Contaminated Ingredients**

For foodborne outbreaks only, indicates the contaminated ingredient. Multiple contaminated ingredients are separated by a semicolon.

In [71]:
outbreak_clean['food_contaminated_ingredient'].nunique()

518

This is a redundant variable, use food vehicle instead. Omit this variable.

**3.  IFSAC Category**

IFSAC stands for the Interagency Food Safety Analytics Collaboration, which is a categorization scheme used to identify foods most often linked to specific illnesses caused by certain pathogens. Source: <https://www.cdc.gov/ifsac/php/projects/food-categorization-scheme.html#:~:text=At%20a%20glance,include%20ice%20and%20dietary%20supplements.>

In [72]:
outbreak_clean['ifsac_category'].nunique()

25

Confirm that this category only exists within Food Vehicle outbreaks and count how many missing values

In [73]:
# Group by primary_mode and check missing ifsac_category
ifsac_summary = outbreak_clean.groupby('primary_mode').agg(
    total=('ifsac_category', 'size'),
    missing_IFSAC=('ifsac_category', lambda x: x.isna().sum())
)

ifsac_summary['percent_missing'] = 100 * ifsac_summary['missing_IFSAC'] / ifsac_summary['total']
ifsac_summary = ifsac_summary.sort_values('percent_missing', ascending=False)

print(ifsac_summary)

                                                   total  missing_IFSAC  \
primary_mode                                                              
Animal contact                                       655            655   
Environmental contamination other than food/water    132            132   
Indeterminate/unknown                               5677           5677   
Person-to-person                                   32414          32414   
Water                                               3101           3101   
Food                                               24727          13023   

                                                   percent_missing  
primary_mode                                                        
Animal contact                                          100.000000  
Environmental contamination other than food/water       100.000000  
Indeterminate/unknown                                   100.000000  
Person-to-person                                      

Looks good, about half of food borne outbreaks contain IFSAC Categories. Might not be much more informative than Food Vehicle for the sake of our study.

**9. Water Exposure**

For waterborne outbreaks only, the implicated type of water exposure.

In [74]:
# Top water exposure types by frequency
water_exposure_counts = (
    outbreak_clean['water_exposure']
    .value_counts(dropna=False)
    .reset_index()
    .rename(columns={'index': 'water_exposure', 'water_exposure': 'count'})
)

water_exposure_counts.head(10)

,count,count
0,NaN,63605
1,Drinking water,1251
2,Recreational water -- treated,1085
3,Recreational water -- untreated,361
4,Undetermined water,192
5,Other/Environmental water,180
6,Drinking water;Recreational water -- treated,14
7,Drinking water;Other/Environmental water,11
8,Drinking water;Undetermined water,5
9,Other/Environmental water;Recreational water -...,1


Investigate where the missing values are coming from

In [75]:
# Create a column flagging missing vs present values
outbreak_clean['water_exposure_missing'] = outbreak_clean['water_exposure'].apply(
    lambda x: "Missing" if pd.isna(x) or str(x).strip() == "" else "Present"
)

# Count by primary_mode and missing/present status
water_missing_summary = (
    outbreak_clean
    .groupby(['primary_mode', 'water_exposure_missing'])
    .size()
    .reset_index(name='count')
)

# Calculate percentages within each primary mode
water_missing_summary['percent'] = (
    water_missing_summary.groupby('primary_mode')['count']
    .transform(lambda x: 100 * x / x.sum())
)

water_missing_summary

,primary_mode,water_exposure_missing,count,percent
0,Animal contact,Missing,655,100.0
1,Environmental contamination other than food/water,Missing,132,100.0
2,Food,Missing,24727,100.0
3,Indeterminate/unknown,Missing,5677,100.0
4,Person-to-person,Missing,32414,100.0
5,Water,Present,3101,100.0


All outbreaks within water as primary mode of exposure have a designated type of water exposure (in other words, no missing values of water exposure in water-borne outbreaks)

**10. Water Type**

For waterborne outbreaks only, a description of the venue (for treated and untreated recreational water), water system (for drinking water) or device/structure (for other/unknown; e.g., steam cleaner, cooling tower, ornamental fountain, etc.) that was the vehicle for waterborne exposure to microbial pathogens, chemicals, or toxins.

In [76]:
# Number of unique water types
outbreak_clean['water_type'].nunique()

55

In [77]:
# Top 10 water types
top_water_types = (
    outbreak_clean['water_type']
    .value_counts(dropna=False)
    .head(10)
    .reset_index()
    .rename(columns={'index': 'water_type', 'water_type': 'count'})
)

top_water_types

,count,count
0,NaN,63981
1,Community,692
2,Pool - Other Swimming Pool,511
3,Hot Tub/Spa/Whirlpool,348
4,Other,345
5,Lake/Reservoir,283
6,Hot Tub/Spa/Whirlpool;Pool - Other Swimming Pool,113
7,Individual/Private,112
8,Unknown,76
9,Splash Pad/Interactive Fountain /Water Playground,41


**11. Animal Type**

For animal contact outbreaks only, the type of animal involved. Multiple animal types are separated by a semicolon.

In [78]:
# Top 10 animal types
top_animal_types = (
    outbreak_clean['animal_type']
    .value_counts(dropna=False)
    .head(10)
    .reset_index()
    .rename(columns={'index': 'animal_type', 'animal_type': 'count'})
)

top_animal_types

,count,count
0,NaN,66119
1,Poultry,184
2,Cattle,147
3,Dog or puppy,46
4,Turtle,41
5,Goat,30
6,Lizard,25
7,Pig,8
8,Hedgehog,7
9,Sheep,7


### B. Numeric Variables (n = 1)

We will only be cleaning illnesses, but more exploratory analysis of the other numeric will be done in a different notebook.

**1. Illnesses**

Estimated total number of primary cases, including lab-confirmed and probable, based on the outbreak-specific definition.

In [79]:
# Remove commas and convert to float
outbreak_clean['illnesses'] = outbreak_clean['illnesses'].astype(str).str.replace(',', '', regex=False)
outbreak_clean['illnesses'] = outbreak_clean['illnesses'].astype(float)

# Check for any null values
print(outbreak_clean['illnesses'].isnull().sum())

# Preview
outbreak_clean[['illnesses']].head()

0


,illnesses
0,2.0
1,98.0
2,2.0
3,3.0
4,3.0


## Handle Missing Data

Remove columns with mostly missing values.

In [80]:
# Drop all unnecessary variables
columns_to_drop = [
    'serotype_or_genotype',
    'info_on_hospitalizations',
    'info_on_deaths',
    'etiology_status',
    'food_contaminated_ingredient'
]

outbreak_clean = outbreak_clean.drop(columns=columns_to_drop)

Impute missing values if necessary (median or “Unknown”).

In [81]:
# Define categorical columns (general)
categorical_cols = ['etiology', 'setting']

# Define contingent categorical columns
contingent_cols = [
    'food_vehicle','ifsac_category', 'water_exposure', 'water_type', 'animal_type'
]

# Fill missing values in categorical columns with 'Unknown'
for col in categorical_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Unknown')

# Fill missing values in contingent columns with 'Not Applicable'
for col in contingent_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna('Not Applicable')

# Handle numeric columns
numeric_cols = ['hospitalizations', 'deaths']
for col in numeric_cols:
    if col in outbreak_clean.columns:
        outbreak_clean[col] = outbreak_clean[col].fillna(outbreak_clean[col].median())

# Check that all missing values are handled
print("Missing values after cleaning:")
print(outbreak_clean.isnull().sum())

Missing values after cleaning:
year                      0
month                     0
state                     0
primary_mode              0
etiology                  0
setting                   0
illnesses                 0
hospitalizations          0
deaths                    0
food_vehicle              0
ifsac_category            0
water_exposure            0
water_type                0
animal_type               0
food_vehicle_missing      0
water_exposure_missing    0
dtype: int64


## Standardize and Normalize Data

Standardize categorical variables (e.g., state abbreviations, illness names).

In [82]:
# Create a 'date' column from 'year' and 'month'
outbreak_clean['date'] = pd.to_datetime(outbreak_clean[['year', 'month']].assign(day=1))

# Add a new column for state abbreviations
def get_state_abbr(state):
    state_info = us.states.lookup(state)
    return state_info.abbr if state_info else 'Unknown'

outbreak_clean['state_abbr'] = outbreak_clean['state'].apply(get_state_abbr)

Normalize numeric variables if you plan to use clustering or distance-based methods using z-score.

In [83]:
# Initialize scaler
scaler = StandardScaler()

# Standardize numeric count columns
outbreak_clean[['illnesses_std', 'hospitalizations_std', 'deaths_std']] = scaler.fit_transform(
    outbreak_clean[['illnesses', 'hospitalizations', 'deaths']]
)

# Preview first 10 rows
outbreak_clean[['illnesses_std', 'hospitalizations_std', 'deaths_std']].head(10)

,illnesses_std,hospitalizations_std,deaths_std
0,-0.019976,-0.184611,-0.091011
1,0.041387,-0.184611,-0.091011
2,-0.019976,-0.184611,-0.091011
3,-0.019336,-0.184611,-0.091011
4,-0.019336,-0.184611,-0.091011
5,0.106584,-0.184611,-0.091011
6,-0.007192,-0.184611,-0.091011
7,0.098274,-0.184611,-0.091011
8,0.022211,-0.184611,-0.091011
9,2.215906,-0.184611,-0.091011


## Feature Engineering

### Etiology Group

We can create a new variable that groups similar etiologies together.

In [84]:
# Create a new grouped column for etiology
conditions = [
    outbreak_clean['etiology'].str.contains('norovirus', case=False, na=False),
    outbreak_clean['etiology'].str.contains('salmonella', case=False, na=False),
    outbreak_clean['etiology'].str.contains('campylobacter', case=False, na=False),
    outbreak_clean['etiology'].str.contains('clostridium', case=False, na=False),
    outbreak_clean['etiology'].str.contains('staphylococcus', case=False, na=False),
    outbreak_clean['etiology'].str.contains('hepatitis', case=False, na=False),
    outbreak_clean['etiology'].str.contains('unknown|unspecified|not determined', case=False, na=False)
]

choices = [
    'Norovirus',
    'Salmonella',
    'Campylobacter',
    'Clostridium',
    'Staphylococcus',
    'Hepatitis',
    'Unknown'
]

# Create new column
outbreak_clean['etiology_grouped'] = np.select(conditions, choices, default='Other')

# Check unique values
outbreak_clean['etiology_grouped'].value_counts()

etiology_grouped
Norovirus         33157
Unknown           17788
Other              7592
Salmonella         4581
Clostridium        1575
Campylobacter      1109
Staphylococcus      757
Hepatitis           147
Name: count, dtype: int64

### Water Type Group

We can create a new variable that groups similar water types together.

In [85]:
# Conditions for grouping
conditions = [
    outbreak_clean['water_type'].isna(),
    outbreak_clean['water_type'].str.contains('community|individual|private', case=False, na=False),
    outbreak_clean['water_type'].str.contains('pool|hot tub|spa|whirlpool|splash pad|fountain|water playground', case=False, na=False),
    outbreak_clean['water_type'].str.contains('lake|reservoir|other', case=False, na=False),
    outbreak_clean['water_type'].str.contains('unknown|unspecified|not determined', case=False, na=False)
]

choices = [
    'Missing/NA',
    'Drinking/Community Water',
    'Recreational Water',
    'Natural/Other',
    'Unknown'
]

# Create new grouped column
outbreak_clean['water_type_group'] = np.select(conditions, choices, default='Other')

# Check counts of the new grouping
outbreak_clean['water_type_group'].value_counts()

water_type_group
Other                       64071
Recreational Water           1060
Drinking/Community Water      855
Natural/Other                 644
Unknown                        76
Name: count, dtype: int64

### Animal Group

We can create a new variable that groups similar animal types together.

In [86]:
# Conditions for grouping
conditions = [
    outbreak_clean['animal_type'].isna(),
    outbreak_clean['animal_type'].str.contains('poultry', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('cattle', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('cat|dog|ferret|guinea pig|hedgehog|mouse|rat|raccoon', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('horse|pony|goat|sheep|pig|alpaca|donkey|llama|yak', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('lizard|turtle|snake|reptile', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('bird, not including poultry', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('fish', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('frog|salamander', case=False, na=False),
    outbreak_clean['animal_type'].str.contains('other', case=False, na=False)
]

choices = [
    'Missing/NA',
    'Poultry',
    'Cattle',
    'House Pets',
    'Other Ungulates',
    'Reptiles',
    'Other Birds',
    'Fish',
    'Amphibians',
    'Other/Unknown'
]

# Create new grouped column
outbreak_clean['animal_group'] = np.select(conditions, choices, default='Other/Unknown')

# Check counts of the new grouping
outbreak_clean['animal_group'].value_counts()

animal_group
Other/Unknown      66125
Poultry              209
Cattle               166
Reptiles              75
House Pets            70
Other Ungulates       56
Fish                   3
Amphibians             2
Name: count, dtype: int64

### Season

Add a varaible that determines what season the outbreak occured in.

In [87]:
def month_to_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Fall'

outbreak_clean['season'] = outbreak_clean['month'].apply(month_to_season)

# Preview
display(outbreak_clean[['month', 'season']].head())

,month,season
0,2,Winter
1,6,Summer
2,6,Summer
3,6,Summer
4,6,Summer


### Severity of Outbreak

Since these following variables are nested within each other illnesses(hospitalizations((deaths))), we'll create 3 new ratios to characterize outbreak severity.

In [92]:
# Compute ratios only when denominator is > 0
outbreak_clean['hosp_per_ill'] = np.where(
    outbreak_clean['illnesses'] > 0,
    outbreak_clean['hospitalizations'] / outbreak_clean['illnesses'],
    np.nan
)

outbreak_clean['death_per_ill'] = np.where(
    outbreak_clean['illnesses'] > 0,
    outbreak_clean['deaths'] / outbreak_clean['illnesses'],
    np.nan
)

# Compute death per hospitalization percentage
outbreak_clean['death_per_hosp'] = np.where(
    outbreak_clean['hospitalizations'] > 0,
    (outbreak_clean['deaths'] / outbreak_clean['hospitalizations']) * 100,
    np.nan
)

# Summaries
print("Mean percentage ratios:")
print(outbreak_clean[['hosp_per_ill', 'death_per_hosp', 'death_per_ill']].mean(skipna=True))

print("\nMissing values:")
print(outbreak_clean[['hosp_per_ill', 'death_per_hosp', 'death_per_ill']].isna().sum())

Mean percentage ratios:
hosp_per_ill      0.053834
death_per_hosp    4.233779
death_per_ill     0.003056
dtype: float64

Missing values:
hosp_per_ill          0
death_per_hosp    51589
death_per_ill         0
dtype: int64


We'll omit death/hospitalizations since it will be hard to handle NAs.

In [93]:
# Drop death_per_hosp
outbreak_clean = outbreak_clean.drop(columns=['death_per_hosp'])

### Feature Flags 

Create flags for food, water, or animal-related outbreaks.

In [94]:
outbreak_clean['is_food_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Food' in x else 0)
outbreak_clean['is_water_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Water' in x else 0)
outbreak_clean['is_animal_related'] = outbreak_clean['primary_mode'].apply(lambda x: 1 if 'Animal' in x else 0)

# Preview
display(outbreak_clean[['primary_mode', 'is_food_related', 'is_water_related', 'is_animal_related']].head())

,primary_mode,is_food_related,is_water_related,is_animal_related
0,Water,0,1,0
1,Water,0,1,0
2,Water,0,1,0
3,Water,0,1,0
4,Water,0,1,0


### Reorder Columns

Clean and reorder the columns for easier use.

In [98]:
outbreak_clean = outbreak_clean[
    [
        # Metadata
        'year', 'month', 'date', 'season', 'state', 'state_abbr',
        
        # Outbreak characteristics
        'primary_mode', 'setting', 'etiology', 'etiology_grouped', 
        'ifsac_category', 'food_vehicle', 'water_exposure', 
        'water_type', 'water_type_group', 'animal_type', 'animal_group',
        
        # Outcome counts
        'illnesses', 'hospitalizations', 'deaths',
        
        # Standardized counts
        'illnesses_std', 'hospitalizations_std', 'deaths_std',
        
        # Ratios
        'hosp_per_ill', 'death_per_ill',
        
        # Missing flags
        'food_vehicle_missing', 'water_exposure_missing',
        
        # Indicators
        'is_food_related', 'is_water_related', 'is_animal_related'
    ]
]

# Preview the cleaned dataset
outbreak_clean.head()

,year,month,date,season,state,state_abbr,primary_mode,setting,etiology,etiology_grouped,...,illnesses_std,hospitalizations_std,deaths_std,hosp_per_ill,death_per_ill,food_vehicle_missing,water_exposure_missing,is_food_related,is_water_related,is_animal_related
0,1971,2,1971-02-01,Winter,California,CA,Water,Restaurant,Copper,Other,...,-0.019976,-0.184611,-0.091011,0.0,0.0,Missing,Present,0,1,0
1,1971,6,1971-06-01,Summer,Arkansas,AR,Water,Store,Hepatitis A,Hepatitis,...,0.041387,-0.184611,-0.091011,0.0,0.0,Missing,Present,0,1,0
2,1971,6,1971-06-01,Summer,Missouri,MO,Water,Subdivision/Neighborhood,Unknown,Unknown,...,-0.019976,-0.184611,-0.091011,0.0,0.0,Missing,Present,0,1,0
3,1971,6,1971-06-01,Summer,Alabama,AL,Water,Unknown,Selenium,Other,...,-0.019336,-0.184611,-0.091011,0.0,0.0,Missing,Present,0,1,0
4,1971,6,1971-06-01,Summer,Vermont,VT,Water,Community/municipality,Unknown,Unknown,...,-0.019336,-0.184611,-0.091011,0.0,0.0,Missing,Present,0,1,0


## Save dataset to CSV

In [99]:
# Save cleaned dataset to CSV
outbreak_clean.to_csv('outbreak_clean_v2.csv', index=False)